# Erdős–Straus Conjecture Solver — Google Colab

**Guinea Pig Trench LLC**

---

The [Erdős–Straus conjecture](https://en.wikipedia.org/wiki/Erd%C5%91s%E2%80%93Straus_conjecture) states that for every integer $n \geq 2$:

$$\frac{4}{n} = \frac{1}{x} + \frac{1}{y} + \frac{1}{z}$$

where $x, y, z$ are positive integers with $x \leq y \leq z$.

We've independently verified this to **100,000,000** using a three-stage pipeline (CPU hunter → CPU leviathan → GPU rescue). This notebook lets you push the frontier further.

Only $n \equiv 1$ or $17 \pmod{24}$ need brute-force — all other residues have known parametric decompositions.

GitHub: [Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)

In [ ]:
# === Setup: detect hardware ===
import os, multiprocessing as mp

cpu_count = mp.cpu_count()
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3) if hasattr(os, 'sysconf') else 0

print(f'CPU cores: {cpu_count}')
print(f'RAM: {ram_gb:.1f} GB')

# GPU check (doesn't help our solver but Colab gives more CPU/RAM to GPU runtimes)
gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null').read().strip()
print(f'GPU: {gpu or "none (CPU runtime)"}')

workers = max(2, cpu_count)
print(f'Workers: {workers}')
print()
print('Tip: Use Runtime > Change runtime type > T4 GPU for more CPU/RAM')

In [ ]:
# === Solver (self-contained, zero dependencies) ===
import math

def solve_single(n, step_cap=20_000_000, y_cap_per_x=2_000_000):
    """Find x <= y <= z such that 4/n = 1/x + 1/y + 1/z."""
    if n <= 1:
        return None
    steps = 0
    x_min = max(1, math.ceil(n / 4))
    x_max = n
    for x in range(x_min, x_max + 1):
        num_r = 4 * x - n
        if num_r <= 0:
            steps += 1
            if steps >= step_cap:
                return None
            continue
        den_r = n * x
        y_min = math.ceil(den_r / num_r)
        y_max = 2 * den_r // num_r
        y_steps = 0
        for y in range(max(x, y_min), y_max + 1):
            steps += 1
            y_steps += 1
            if steps >= step_cap:
                return None
            if y_steps >= y_cap_per_x:
                break
            denom_z = num_r * y - den_r
            if denom_z <= 0:
                continue
            num_z = den_r * y
            if num_z % denom_z == 0:
                z = num_z // denom_z
                if z >= y:
                    return {'n': n, 'x': x, 'y': y, 'z': z, 'steps': steps}
    return None

def worker(args):
    n, cap = args
    r = solve_single(n, step_cap=cap)
    if r:
        r['solved'] = True
        return r
    return {'n': n, 'x': 0, 'y': 0, 'z': 0, 'steps': cap, 'solved': False}

# Sanity check
t = solve_single(5)
assert t and t['x'] == 2 and t['y'] == 4 and t['z'] == 20
print(f'Solver loaded. Sanity: 4/5 = 1/{t["x"]} + 1/{t["y"]} + 1/{t["z"]}')

---
## Configure Your Range

Edit the cell below. Only hard-residue targets (n mod 24 ∈ {1, 17}) are generated.

In [ ]:
# === Set your range ===
n_start = 100_000_001   # <-- edit this
n_end   = 101_000_000   # <-- edit this (1M range = ~83K targets)
step_cap = 20_000_000   # increase to 100M for stubborn holdouts

ns = [n for n in range(n_start, n_end + 1) if n % 24 in (1, 17)]
print(f'Range: {n_start:,} to {n_end:,}')
print(f'Hard-residue targets: {len(ns):,}')
print(f'Step cap: {step_cap:,}')

In [ ]:
# === OR: Upload a chunk file instead ===
# Uncomment these lines to upload specific n values:

# from google.colab import files
# uploaded = files.upload()
# chunk_file = list(uploaded.keys())[0]
# with open(chunk_file) as f:
#     ns = [int(line.strip()) for line in f if line.strip()]
# print(f'Loaded {len(ns):,} values from {chunk_file}')

In [ ]:
# === Run the Solver (auto-resume supported) ===
import csv, time

out_file = f'erdos_results_{n_start}_{n_end}.csv'
fields = ['n', 'x', 'y', 'z', 'steps', 'solved']

# Auto-resume: skip already-solved
done = set()
if os.path.exists(out_file):
    with open(out_file) as f:
        for row in csv.DictReader(f):
            done.add(int(row['n']))
    print(f'Resuming: {len(done):,} already done')

remaining = [n for n in ns if n not in done]
if not remaining:
    print('All done! Nothing to solve.')
else:
    total = len(remaining)
    batch_size = workers * 8
    solved_count = len(done)
    unsolved_count = 0
    all_results = []
    mode = 'a' if done else 'w'
    t0 = time.time()

    print(f'Solving {total:,} values | workers={workers} | step_cap={step_cap:,}')
    print('=' * 60)

    with open(out_file, mode, newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if not done:
            writer.writeheader()

        with mp.Pool(workers) as pool:
            for i in range(0, total, batch_size):
                batch = remaining[i:i + batch_size]
                results = pool.map(worker, [(n, step_cap) for n in batch])

                for r in results:
                    writer.writerow(r)
                    all_results.append(r)
                    if r['solved']:
                        solved_count += 1
                    else:
                        unsolved_count += 1

                processed = i + len(batch)
                if processed % (batch_size * 50) < batch_size or processed >= total:
                    f.flush()
                    elapsed = time.time() - t0
                    rate = processed / elapsed
                    eta = (total - processed) / rate if rate > 0 else 0
                    print(f'  [{100*processed/total:5.1f}%] {processed:,}/{total:,} | '
                          f'solved: {solved_count:,} | unsolved: {unsolved_count} | '
                          f'{rate:.0f}/s | ETA: {eta/60:.0f}m')

    total_time = time.time() - t0
    print('=' * 60)
    print(f'Done in {total_time/60:.1f}m | Solved: {solved_count:,} | Unsolved: {unsolved_count}')
    print(f'Output: {out_file}')

In [ ]:
# === Save to Google Drive ===
import shutil
from google.colab import drive, files

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/erdos_straus'
os.makedirs(drive_dir, exist_ok=True)
shutil.copy2(out_file, os.path.join(drive_dir, out_file))
print(f'Saved to Drive: {drive_dir}/{out_file}')

# Also download locally
files.download(out_file)

In [ ]:
# === Stats ===
if 'all_results' in dir() and all_results:
    solved = [r for r in all_results if r['solved']]
    unsolved = [r for r in all_results if not r['solved']]

    print(f'Total: {len(all_results):,} | Solved: {len(solved):,} | Unsolved: {len(unsolved):,}')

    if solved:
        max_z_row = max(solved, key=lambda r: r['z'])
        print(f'Largest z: {max_z_row["z"]:,} (n={max_z_row["n"]:,})')
        print(f'Avg steps: {sum(r["steps"] for r in solved)/len(solved):,.0f}')
        print(f'Max steps: {max(r["steps"] for r in solved):,}')

    if unsolved:
        print(f'\nUnsolved values (need higher step_cap):')
        for r in unsolved[:20]:
            print(f'  n = {r["n"]:,}')
    else:
        print('\nAll values solved! The conjecture holds for this range.')
else:
    print('No results yet — run the solver cell first.')